# 05 · Reranking: cross-encoders, LLM rerankers and MMR

After this notebook you can add a second, precision-oriented stage to retrieval: rerank the top 20 hybrid candidates with a cross-encoder, measure the MRR gain on labelled questions, explain why the reranker cannot rescue a document the first stage missed, estimate its latency budget, use an LLM as a listwise reranker with validated structured output, and remove redundancy with MMR.

**Time:** ~25 min · **Runs:** offline on CPU in ~1 min (two 90 MB models, cached) · **Needs:** [hybrid search and RRF](04_hybrid_search_and_rrf.ipynb)

## Why this matters in interviews

- "What is a reranker and why not use it for retrieval?" is a standard pairing. The answer is the bi-encoder vs cross-encoder trade-off: precomputable and fast vs joint attention and accurate.
- The trap interviewers set: *"retrieval quality is bad, so we added a reranker"*. A reranker only reorders what it is given - if recall is bad, it spends latency reordering the wrong documents. You will demonstrate that.
- Latency: the reranker is often the largest retrieval term in a RAG latency budget. Knowing "ms per pair × candidates" lets you size it on the spot.

## Interview questions this notebook answers

| ID | Question |
|----|----------|
| `rg08` | What is reranking, and why can a reranker not fix bad recall? |
| `ve17` | What is a cross-encoder and why can it not be used for first-stage retrieval? |
| `ve25` | What is the difference between an embedding model and a reranker model? |
| `ve31` | What is the difference between semantic similarity and relevance? |
| `ve29` | What is recall@k, MRR and nDCG - and when do you use each? |
| `ve24` | What is MMR (maximal marginal relevance) and when do you want it? |
| `rg27` | How many chunks should you retrieve, and does more always help? |
| `rg48` | What does a RAG latency budget look like? |

In [ ]:
import sys
from pathlib import Path
for _p in [Path.cwd(), *Path.cwd().parents]:
    if (_p / "llm_setup.py").exists():
        sys.path.insert(0, str(_p)); break
from llm_setup import client, MODEL, EMBED_MODEL, PROVIDER

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import FancyBboxPatch
from rank_bm25 import BM25Okapi

np.random.seed(0)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 60)

In [ ]:
# The knowledge base: 15 short markdown docs for a fictional company, Acme Cloud (a SaaS analytics product
# that also sells an 'Edge Gateway' device). This cell is data - it is identical in every 09_rag notebook.
KB = {
"refund_policy.md": """# Refund Policy

## Eligibility
Annual plans can be refunded in full within 30 days of purchase. Monthly plans can be cancelled at any time, but the current month is not refunded. Add-on credits and professional-services hours are non-refundable.

## How to request a refund
Open the billing portal, choose the invoice and click "Request refund". Only workspace owners can request refunds.

## Processing time
Approved refunds go back to the original payment method within 5-10 business days. Bank transfers can take up to 15 business days.
""",
"shipping.md": """# Hardware Shipping

## Delivery times
Edge Gateway devices ship from our warehouses in Rotterdam and Dallas. Standard delivery takes 3-5 business days. Express delivery takes 1-2 business days and costs $25.

## Tracking
A tracking link is emailed when the order leaves the warehouse. The first carrier scan can take 24 hours to appear.

## Lost or damaged shipments
If a shipment has not arrived 10 business days after dispatch, report it within 14 days so we can open a carrier investigation. We send a free replacement when the carrier confirms the loss.
""",
"warranty.md": """# Edge Gateway Warranty

## Coverage
Every Edge Gateway has a 2-year limited hardware warranty from the delivery date. It covers manufacturing defects in the device, the power supply and the antennas.

## Not covered
Water damage, drops, unauthorised repairs and cosmetic scratches are not covered by the warranty.

## RMA process
Request an RMA (return merchandise authorisation) in the support portal with the device serial number. Once the RMA is approved we ship a replacement within 7 days, and you send the faulty unit back with the prepaid label.
""",
"api_rate_limits.md": """# API Rate Limits

## Limits per plan
Free: 60 requests per minute. Pro: 600 requests per minute. Business: 1,200 requests per minute. Enterprise: 6,000 requests per minute. Limits apply per API key.

## Best practices
Retry with exponential backoff and jitter, cache responses you read often, and send writes through the bulk endpoint.
""",
"api_errors.md": """# API Error Reference

## ERR-429-A
The per-minute rate limit was exceeded. Wait for the number of seconds in the Retry-After header, then retry.

## ERR-429-B
The daily request quota is exhausted. The quota resets at 00:00 UTC; upgrade the plan for a higher quota.

## ERR-503-C
The API is in scheduled maintenance. Retry after the maintenance window shown on the status page.

## ERR-401-D
The API key was revoked or has expired. Create a new key in Settings > API keys.
""",
"gateway_models.md": """# Edge Gateway Models

## EG-100
The EG-100 is the entry model: 2 Ethernet ports, Wi-Fi 5 and an 8 W maximum power draw. It supports up to 50 sensors.

## EG-200
The EG-200 adds LTE backup: 4 Ethernet ports, Wi-Fi 6 and a 12 W maximum power draw. It supports up to 200 sensors.

## EG-300
The EG-300 is the rugged outdoor model: an IP67 enclosure, 4 Ethernet ports, Wi-Fi 6 and an 18 W maximum power draw. It supports up to 500 sensors.
""",
"oncall_policy.md": """# On-call Policy

## Rotation
Engineers join the on-call rotation after their third month. Each shift lasts one week and hands over on Monday at 09:00 local time. Every shift has a primary and a secondary.

## Response targets
A SEV1 page must be acknowledged within 5 minutes and a SEV2 page within 30 minutes. If a SEV1 page is not acknowledged within 15 minutes, it escalates to the secondary and then to the engineering manager.

## Compensation
On-call engineers receive a stipend of $300 per week, and a day off after any night with more than two pages.
""",
"pricing.md": """# Pricing Plans

## Plans
Free costs $0 and includes 3 users and 1 GB of storage. Pro costs $49 per user per month. Business costs $99 per user per month. Enterprise pricing is custom and starts at 50 users.

## What each plan includes
Single sign-on (SAML and OIDC) is included from the Business plan upward. Audit logs and the 99.95% uptime SLA are included only in Enterprise. All paid plans include email support.

## Billing
Paying annually saves 20% compared with monthly billing. Prices exclude VAT.
""",
"sso_setup.md": """# Single Sign-On Setup

## Supported identity providers
Acme Cloud supports Okta, Microsoft Entra ID (formerly Azure AD) and Google Workspace through SAML 2.0 or OIDC.

## Configuration steps
Go to Settings > Security > Single sign-on, upload the identity provider's metadata XML, and map the email and groups attributes. Test with one account before you enforce SSO for the whole workspace.

## Troubleshooting
SSO-ERR-12 means the SAML assertion was rejected because the clocks differ by more than 5 minutes; sync the identity provider's clock with NTP. SSO-ERR-31 means the email attribute is missing from the assertion.
""",
"data_retention.md": """# Data Retention

## Event data
Raw event data is kept for 30 days on Free, 13 months on Pro and Business, and 25 months on Enterprise. Aggregated reports are kept for as long as the workspace exists.

## Deleted workspaces
When a workspace is deleted, its data is purged from primary storage after 30 days and from backups after a further 35 days.

## Erasure requests
GDPR erasure requests for one person's data are completed within 30 days of identity verification.
""",
"pto_policy.md": """# Paid Time Off

## Allowance
Full-time employees receive 25 days of paid time off per calendar year, plus public holidays. Part-time employees receive a pro-rata allowance.

## Booking
Book absences longer than three days in Workday at least two weeks ahead. Your manager approves the request.

## Carry-over
Up to 5 unused days can be carried into the next year and must be used by 31 March.
""",
"webhooks.md": """# Webhooks

## Delivery and retries
Webhook events are sent as HTTPS POST requests. If your endpoint does not return a 2xx status within 10 seconds, we retry up to 8 times with exponential backoff over 24 hours.

## Verifying signatures
Every request carries an X-Acme-Signature header: an HMAC-SHA256 of the raw body computed with your webhook secret. Compare it in constant time and reject requests older than 5 minutes.
""",
"sla.md": """# Service Level Agreement

## Uptime commitment
Pro and Business are backed by a 99.9% monthly uptime commitment. Enterprise is backed by 99.95%.

## Service credits
If monthly uptime falls below the commitment, you receive a service credit of 10% of the monthly fee; below 99.0% the credit is 25%. Credits are applied to the next invoice.

## How to claim
Open a support ticket within 30 days of the end of the affected month and include the incident dates.
""",
"account_security.md": """# Account Security

## Passwords
Password reset links expire after 60 minutes. After 5 failed sign-in attempts the account is locked for 15 minutes.

## Two-factor authentication
Workspace owners can require two-factor authentication (2FA) for every member. Authenticator apps and hardware security keys are supported; SMS codes are not.
""",
"expenses.md": """# Travel and Expenses

## Meals and hotels
Meals are reimbursed up to $75 per day while travelling. Book hotels through the travel portal; the nightly cap is $200 in most cities.

## Submitting claims
Submit receipts in Workday within 30 days of the expense. Claims over $25 without a receipt are rejected.
""",
}

In [ ]:
# The eval set: each question lists its EVIDENCE - (doc, phrase) pairs that a retrieved chunk must contain.
# Phrases are unique in the KB, so "is the evidence in the retrieved text?" works for any chunking scheme.
QUESTIONS = [
    {"q": "What does ERR-429-A mean?", "kind": "exact id", "evidence": [("api_errors.md", "per-minute rate limit")]},
    {"q": "Our integration keeps failing with ERR-401-D", "kind": "exact id", "evidence": [("api_errors.md", "revoked")]},
    {"q": "How do I fix SSO-ERR-12?", "kind": "exact id", "evidence": [("sso_setup.md", "clocks differ")]},
    {"q": "How many sensors does the EG-300 support?", "kind": "exact id", "evidence": [("gateway_models.md", "500 sensors")]},
    {"q": "How do I get my money back after buying an annual plan by mistake?", "kind": "paraphrase",
     "evidence": [("refund_policy.md", "refunded in full within 30 days")]},
    {"q": "My parcel never showed up. What should I do?", "kind": "paraphrase", "evidence": [("shipping.md", "report it within 14 days")]},
    {"q": "How many vacation days do I get a year?", "kind": "paraphrase", "evidence": [("pto_policy.md", "25 days of paid time off")]},
    {"q": "Do we get compensated for outages?", "kind": "paraphrase", "evidence": [("sla.md", "service credit")]},
    {"q": "Which gateway can be installed outdoors?", "kind": "paraphrase", "evidence": [("gateway_models.md", "rugged outdoor model")]},
    {"q": "Who gets paged when nobody acknowledges a SEV1 alert?", "kind": "keyword", "evidence": [("oncall_policy.md", "engineering manager")]},
    {"q": "How long is event data kept on the Pro plan?", "kind": "keyword", "evidence": [("data_retention.md", "13 months on Pro")]},
    {"q": "What do we get back if the platform is down more than promised?", "kind": "vague", "evidence": [("sla.md", "service credit")]},
    {"q": "What is the API rate limit on the cheapest plan that includes SSO?", "kind": "multi-hop",
     "evidence": [("pricing.md", "from the Business plan upward"), ("api_rate_limits.md", "Business: 1,200 requests")]},
    {"q": "How long do approved refunds take, and how long is the gateway warranty?", "kind": "multi-part",
     "evidence": [("refund_policy.md", "5-10 business days"), ("warranty.md", "2-year")]},
    {"q": "What is the capital of Australia?", "kind": "out-of-scope", "evidence": []},
]
IN_SCOPE = [q for q in QUESTIONS if q["evidence"]]


def has_evidence(text, doc, q):
    """True if this text (from `doc`) contains any of the question's evidence phrases."""
    return any(doc == d and p.lower() in text.lower() for d, p in q["evidence"])


print(f"{len(KB)} docs, {sum(len(t) for t in KB.values()):,} characters; "
      f"{len(QUESTIONS)} questions ({len(IN_SCOPE)} answerable, {len(QUESTIONS) - len(IN_SCOPE)} out of scope)")

In [ ]:
import re


def split_sections(kb):
    """One chunk per '## ' section. The heading path is prepended so every chunk says where it came from."""
    chunks = []
    for doc, text in kb.items():
        title = text.splitlines()[0].lstrip("# ").strip()
        for i, part in enumerate(re.split(r"^## ", text, flags=re.M)[1:]):
            heading, _, body = part.partition("\n")
            path = f"{title} > {heading.strip()}"
            chunks.append({"id": f"{doc}#{i}", "doc": doc, "section": path, "text": f"{path}\n{body.strip()}"})
    return chunks


def is_relevant(chunk, q):
    return has_evidence(chunk["text"], chunk["doc"], q)


CHUNKS = split_sections(KB)
print(f"{len(CHUNKS)} section chunks, avg {sum(len(c['text']) for c in CHUNKS) / len(CHUNKS):.0f} characters")
print("example:", CHUNKS[1]["id"], "->", CHUNKS[1]["text"][:90].replace("\n", " | "), "...")

In [ ]:
import os
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
import transformers
transformers.utils.logging.disable_progress_bar()          # keep the cell output clean
transformers.logging.set_verbosity_error()
from sentence_transformers import SentenceTransformer


def load_st(name, cls=SentenceTransformer):
    """Load a small Hugging Face model: cached copy first (no network), download only if missing."""
    try:
        return cls(name, local_files_only=True)
    except Exception:
        try:
            return cls(name)                                 # first run only: ~90 MB download
        except Exception as e:
            raise RuntimeError(f"Could not load {name}: connect to the internet once so it can download.") from e


embedder = load_st("sentence-transformers/all-MiniLM-L6-v2")


def encode(texts):
    """Unit-length MiniLM embeddings (384-d), so a dot product IS the cosine similarity."""
    return embedder.encode(list(texts), normalize_embeddings=True, convert_to_numpy=True)


print("embedder: all-MiniLM-L6-v2 |", embedder.get_embedding_dimension(), "dims | max",
      embedder.max_seq_length, "tokens per text (the rest is silently cut off)")

## 1. Two-stage retrieval

**In plain English:** the first stage is a fast, generous net (recall: "is the answer somewhere in these 20?"); the second stage is a slow, careful judge (precision: "which of these 20 actually answers the question?"). Only the best few go to the LLM.

In [ ]:
def draw_pipeline(steps, title=None):
    """Boxes joined by arrows, left to right."""
    fig, ax = plt.subplots(figsize=(min(2.3 * len(steps), 11), 1.7))
    for i, s in enumerate(steps):
        x = i * 2.3
        ax.add_patch(FancyBboxPatch((x, 0), 1.9, 1, boxstyle="round,pad=0.06", fc="#e7f0ff", ec="#3b5bdb"))
        ax.text(x + 0.95, 0.5, s, ha="center", va="center", fontsize=9)
        if i < len(steps) - 1:
            ax.annotate("", xy=(x + 2.28, 0.5), xytext=(x + 1.98, 0.5), arrowprops=dict(arrowstyle="->", lw=1.6))
    ax.set_xlim(-0.2, len(steps) * 2.3); ax.set_ylim(-0.25, 1.25); ax.axis("off")
    if title: ax.set_title(title)
    plt.show()


draw_pipeline(["question", "hybrid retrieval\n(BM25 + dense)\ntop 20", "cross-encoder\nscores 20 pairs",
               "top 5", "LLM prompt"], title="Retrieve wide (cheap, recall) -> rerank narrow (expensive, precision)")

### Bi-encoder vs cross-encoder

```
 BI-ENCODER (retrieval)                         CROSS-ENCODER (reranking)
 query ──► [encoder] ──► q-vector ─┐            [CLS] query [SEP] passage [SEP]
                                    ├─ cosine              │
 passage ─► [encoder] ─► p-vector ─┘                 [one encoder, full attention
 (passage vectors computed ONCE, offline)             between query and passage words]
                                                           │
                                                     relevance score
                                                  (one forward pass PER PAIR, at query time)
```

| | Bi-encoder (embedding model) | Cross-encoder (reranker) |
|---|---|---|
| Encodes | query and passage **separately** | query and passage **together** |
| Precomputable | yes - index once | no - runs per (query, passage) pair |
| Speed | millions of passages in milliseconds (ANN) | tens to hundreds of pairs per query |
| Accuracy | good: similarity | better: trained on **relevance** |
| Role | recall - get the answer into the candidate set | precision - put it at the top |
| Can it fix the other's failure? | no | no - it never sees what retrieval missed |

## 2. The first stage (hybrid, from notebook 04)

In [ ]:
STOPWORDS = set("""a an the is are was were be been of to in on for with and or but if then so as at by from this that
it its i you we my our your do does did can how what who when which why there their they me""".split())


def tokenize(text):
    return [t for t in re.findall(r"[a-z0-9]+(?:-[a-z0-9]+)*", text.lower()) if t not in STOPWORDS]


bm25 = BM25Okapi([tokenize(c["text"]) for c in CHUNKS])
chunk_vecs = encode([c["text"] for c in CHUNKS])


def rrf(ranked_lists, k=60):
    scores = {}
    for lst in ranked_lists:
        for rank, doc in enumerate(lst, start=1):
            scores[doc] = scores.get(doc, 0.0) + 1.0 / (k + rank)
    return sorted(scores, key=lambda d: -scores[d])


def first_stage(query, depth=10):
    """Hybrid candidates: RRF of BM25 top-10 (matches only) and dense top-10 -> at most 20 chunk ids."""
    b = bm25.get_scores(tokenize(query))
    b_ids = [i for i in np.argsort(-b, kind="stable") if b[i] > 0][:depth]
    d_ids = list(np.argsort(-(chunk_vecs @ encode([query])[0]), kind="stable")[:depth])
    return rrf([b_ids, d_ids])


def first_hit(order, q):
    return next((r for r, i in enumerate(order, 1) if is_relevant(CHUNKS[i], q)), None)


cands = first_stage("Which gateway can be installed outdoors?")
print(f"{len(cands)} candidates; first-stage top 5:", [CHUNKS[i]["id"] for i in cands[:5]])

## 3. Rerank the candidates with a cross-encoder

`cross-encoder/ms-marco-MiniLM-L6-v2` is a 22M-parameter BERT-style model trained on MS MARCO (web search queries and passages) to output a relevance logit for a (query, passage) pair. Higher = more relevant; the scale is arbitrary (apply a sigmoid for 0-1).

In [ ]:
from sentence_transformers import CrossEncoder

reranker = load_st("cross-encoder/ms-marco-MiniLM-L6-v2", cls=CrossEncoder)


_ce_cache = {}                        # (query, chunk id) -> score, so later cells never re-score a pair


def ce_scores(query, cand_ids):
    todo = [i for i in cand_ids if (query, i) not in _ce_cache]
    if todo:
        for i, s in zip(todo, reranker.predict([(query, CHUNKS[i]["text"]) for i in todo])):
            _ce_cache[(query, i)] = float(s)
    return np.array([_ce_cache[(query, i)] for i in cand_ids])


def rerank(query, cand_ids, top_n=5):
    scores = ce_scores(query, cand_ids)
    order = np.argsort(-scores, kind="stable")
    return [cand_ids[j] for j in order][:top_n], scores[order][:top_n]


query = "Which gateway can be installed outdoors?"
ids, scores = rerank(query, cands)
print(query)
print("before (hybrid):", [CHUNKS[i]["id"] for i in cands[:5]])
print("after (rerank): ", [f"{CHUNKS[i]['id']} ({s:.1f})" for i, s in zip(ids, scores)])
assert is_relevant(CHUNKS[ids[0]], QUESTIONS[8]) and first_hit(cands, QUESTIONS[8]) > 1

The EG-300 chunk ("the rugged outdoor model") moves from 3rd to 1st. The cross-encoder reads "installed outdoors" and "rugged outdoor model ... IP67 enclosure" *together*, instead of comparing two vectors computed apart. Now across the whole eval set:

In [ ]:
rows = []
for q in IN_SCOPE:
    cands = first_stage(q["q"])
    reranked, _ = rerank(q["q"], cands, top_n=len(cands))
    rows.append({"kind": q["kind"], "question": q["q"][:55], "in candidates?": first_hit(cands, q) is not None,
                 "rank before": first_hit(cands, q), "rank after": first_hit(reranked, q)})
res = pd.DataFrame(rows)
print(res.fillna("-").to_string(index=False))


def mrr(ranks):
    return np.mean([1 / r if r == r and r else 0 for r in ranks])      # r == r filters NaN


def hit1(ranks):
    return np.mean([r == 1 for r in ranks])


print(f"\nMRR   before {mrr(res['rank before']):.3f} -> after {mrr(res['rank after']):.3f}")
print(f"hit@1 before {hit1(res['rank before']):.3f} -> after {hit1(res['rank after']):.3f}")
comp = next(q for q in IN_SCOPE if q["q"].startswith("Do we get compensated"))
print(f"reranker's #1 for {comp['q']!r}: {CHUNKS[rerank(comp['q'], first_stage(comp['q']), 1)[0][0]]['id']}")
assert mrr(res["rank after"]) > mrr(res["rank before"]) and hit1(res["rank after"]) > hit1(res["rank before"])

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
y = np.arange(len(res))
for yi, (_, r) in zip(y, res.iterrows()):
    if r["in candidates?"]:
        color = "#2b8a3e" if r["rank after"] < r["rank before"] else ("#c92a2a" if r["rank after"] > r["rank before"] else "#868e96")
        ax.annotate("", xy=(r["rank after"], yi), xytext=(r["rank before"], yi), arrowprops=dict(arrowstyle="->", color=color, lw=2))
        ax.plot(r["rank before"], yi, "o", color="#adb5bd")
    else:
        ax.text(1, yi, "not in the 20 candidates - the reranker never sees it", va="center", color="#c92a2a", fontsize=8)
ax.set_yticks(y)
ax.set_yticklabels(res["question"], fontsize=8)
ax.invert_yaxis()
ax.set_xlabel("rank of the first chunk holding the evidence (1 = top)")
ax.set_xlim(0.5, 8)
ax.set_title("Before (grey dot) -> after reranking (arrow head)")
plt.tight_layout()
plt.show()

Most arrows point left or stay at 1. Read the exceptions too:

- The **red arrow** ("Do we get compensated for outages?") is a question where the reranker *demoted* the evidence: it was trained on web search, not on Acme's support vocabulary, and "compensated" pulls it toward the on-call compensation section. Rerankers are better on average, not on every query - measure on your data.
- **The vague question** never reached the candidate set, so the reranker could not help. That is `rg08` in one row:

In [ ]:
vague = next(q for q in IN_SCOPE if q["kind"] == "vague")
top20 = first_stage(vague["q"])
everything = list(range(len(CHUNKS)))
print("vague question:", vague["q"])
print("  evidence in first-stage top 20?          ", first_hit(top20, vague) is not None)
print("  rank after reranking the top 20:          ", first_hit(rerank(vague["q"], top20, top_n=20)[0], vague))
print("  rank if we rerank ALL 42 chunks instead:  ", first_hit(rerank(vague["q"], everything, top_n=42)[0], vague))
assert first_hit(top20, vague) is None

Given the whole corpus the cross-encoder does find the evidence - but "rerank everything" is not an option on a real corpus. The fix belongs upstream: better recall (query rewriting, [notebook 07](07_advanced_rag_query_rewriting_hyde_multiquery.ipynb)) or a deeper candidate pool. **Measure recall@candidates before you buy a reranker.**

## 4. Latency: why you cannot rerank the whole corpus

In [ ]:
def best_time(fn, repeats=3):
    best = float("inf")
    for _ in range(repeats):
        t0 = time.perf_counter()
        fn()
        best = min(best, time.perf_counter() - t0)
    return best


q = IN_SCOPE[0]["q"]
sizes = [5, 10, 20]
lat = [1000 * best_time(lambda n=n: reranker.predict([(q, CHUNKS[i]["text"]) for i in range(n)]), repeats=2) for n in sizes]
embed_ms = 1000 * best_time(lambda: encode([q]))
per_pair = lat[-1] / sizes[-1]
for n, ms in zip(sizes, lat):
    print(f"rerank {n:3d} candidates: {ms:7.1f} ms")
print(f"embed the query once (bi-encoder side): {embed_ms:.1f} ms")
print(f"~{per_pair:.2f} ms per pair on this CPU -> reranking 1,000,000 chunks would take ~{per_pair * 1e6 / 1000 / 60:.0f} minutes per query")
assert per_pair * 1e6 / 1000 > 60, "a cross-encoder over a million chunks is far beyond any latency budget"

plt.figure(figsize=(7, 3.4))
plt.plot(sizes, lat, "o-", color="#3b5bdb")
plt.xlabel("candidates reranked")
plt.ylabel("ms (best of 3, CPU)")
plt.title("Cross-encoder cost grows linearly with the candidate count")
plt.tight_layout()
plt.show()

Numbers vary by machine (a GPU is 10-50× faster), but the shape does not: cost = candidates × one forward pass. The bi-encoder side is one query embedding plus an ANN lookup, independent of corpus size. In a RAG latency budget, a reranker over 50-100 candidates is typically **50-400 ms** - often the single largest retrieval term - so the candidate count is the dial.

**How many chunks to keep after reranking (`rg27`)?** 3-5 is typical. More chunks raise recall and lower context precision, cost tokens, and bury the answer in the middle of the prompt.

## 5. An LLM as the reranker (listwise)

Instead of scoring pairs, give an LLM the question and all candidates at once and ask for the ids in order - a **listwise** reranker. Use structured output so the reply is machine-readable, then **validate it**: drop ids you never sent, drop duplicates, append anything the model forgot in first-stage order. Never trust ids from a model.

In [ ]:
from pydantic import BaseModel, Field


class Ranking(BaseModel):
    ids: list[int] = Field(description="passage ids, most relevant first")


def llm_rerank(query, cand_ids, top_n=5):
    passages = "\n".join(f"[{j}] {CHUNKS[i]['text'].replace(chr(10), ' ')}" for j, i in enumerate(cand_ids))
    resp = client.chat.completions.parse(
        model=MODEL,
        messages=[{"role": "system", "content": "Rank the passages by how well they answer the question. "
                                                "Return every passage id, most relevant first."},
                  {"role": "user", "content": f"Question: {query}\n\nPassages:\n{passages}"}],
        response_format=Ranking,
        temperature=0,
    )
    raw = resp.choices[0].message.parsed.ids
    valid = list(dict.fromkeys(j for j in raw if 0 <= j < len(cand_ids)))           # known, deduplicated
    merged = valid + [j for j in range(len(cand_ids)) if j not in valid]            # forgotten ids keep their order
    return [cand_ids[j] for j in merged][:top_n], raw


cands = first_stage(query)[:8]
top, raw = llm_rerank(query, cands)
print(f"provider={PROVIDER} | model returned ids: {raw}")
print("after validation + merge:", [CHUNKS[i]["id"] for i in top])
assert len(top) == 5 and len(set(top)) == 5 and set(top) <= set(cands)

With a real model (`gpt-4.1-mini`, `qwen2.5:3b` via Ollama) the reply is a full permutation such as `[2, 0, 5, 1, ...]` with the EG-300 passage first. The **offline stand-in** just fills the schema from numbers it sees in the prompt, so it returns one id - which is exactly why the validation step exists: the pipeline still produces five valid, distinct candidates.

| | Cross-encoder | LLM listwise reranker |
|---|---|---|
| Cost / latency | ~ms per pair, local | one LLM call; prompt grows with candidates × passage length |
| Quality | strong, trained for relevance | can be stronger on reasoning-heavy or domain questions |
| Failure modes | domain shift | invented or missing ids, position bias, truncation on long lists |
| Calibration | a score per pair (usable as a confidence signal) | only an order |

## 6. MMR: relevant *and* different

When chunks overlap heavily (large chunk overlap, versioned documents, boilerplate), the top 5 can be the same sentence five times - wasted context. **Maximal Marginal Relevance** picks greedily:

$$\text{MMR} = \arg\max_{d \notin S} \Big[\lambda \cdot \text{sim}(q, d) - (1 - \lambda) \cdot \max_{s \in S} \text{sim}(d, s)\Big]$$

$\lambda = 1$ is pure relevance; lower $\lambda$ buys diversity. Build a deliberately redundant index - 160-character sliding windows every 40 characters, so each sentence appears in about four overlapping chunks - to see it.

In [ ]:
ov_chunks = []
for name, body in KB.items():
    text = " ".join(line for line in body.splitlines() if line and not line.startswith("#"))
    for start in range(0, max(1, len(text) - 100), 40):
        ov_chunks.append((text[start:start + 160], name))
ov_vecs = encode([t for t, _ in ov_chunks])
print(f"{len(ov_chunks)} overlapping windows")


def mmr(query_vec, doc_vecs, k=5, lam=0.5):
    rel = doc_vecs @ query_vec
    selected = [int(np.argmax(rel))]
    while len(selected) < k:
        redundancy = (doc_vecs @ doc_vecs[selected].T).max(axis=1)
        score = lam * rel - (1 - lam) * redundancy
        score[selected] = -np.inf
        selected.append(int(np.argmax(score)))
    return selected


def redundancy(ids):
    sims = ov_vecs[ids] @ ov_vecs[ids].T
    return float(sims[np.triu_indices(len(ids), 1)].mean())


mq = "How long do approved refunds take?"
qv = encode([mq])[0]
plain = list(np.argsort(-(ov_vecs @ qv))[:5])
diverse = mmr(qv, ov_vecs, k=5, lam=0.7)
for name, ids in [("top-5 by relevance", plain), ("MMR, lambda = 0.7", diverse)]:
    print(f"\n{name}: mean pairwise similarity between the picks = {redundancy(ids):.3f}")
    for i in ids:
        print(f"   {ov_vecs[i] @ qv:.3f} {ov_chunks[i][1]:18s} {ov_chunks[i][0][:70]!r}")
assert redundancy(diverse) < redundancy(plain)

In [ ]:
lams = np.linspace(0.1, 1.0, 10)
rel_curve, red_curve = [], []
for lam in lams:
    ids = mmr(qv, ov_vecs, k=5, lam=lam)
    rel_curve.append(float(np.mean(ov_vecs[ids] @ qv)))
    red_curve.append(redundancy(ids))
plt.figure(figsize=(8, 3.6))
plt.plot(lams, rel_curve, "o-", color="#3b5bdb", label="mean relevance to the query")
plt.plot(lams, red_curve, "s-", color="#e8590c", label="mean similarity between picks (redundancy)")
plt.xlabel("lambda (1 = pure relevance)")
plt.ylabel("cosine")
plt.title("MMR trades a little relevance for a lot less redundancy")
plt.legend()
plt.tight_layout()
plt.show()

Pure relevance spends most of the five slots on overlapping windows of the same refund paragraphs; MMR at λ = 0.7 swaps some of them for different content, lowering redundancy while giving up a little relevance. The curve shows the catch: push λ much lower and MMR starts filling slots with *irrelevant* chunks just because they are different (relevance falls off a cliff). λ is a dial to tune on your eval set, usually 0.5-0.8. The honest alternative (`ve24` trap): if the redundancy comes from your own chunk overlap, **fix the overlap or deduplicate** (by content hash, or by parent id) rather than tuning λ. LangChain vector stores expose it as `store.as_retriever(search_type="mmr", search_kwargs={"k": 5, "fetch_k": 20, "lambda_mult": 0.5})`.

## 7. Hosted rerankers

The same second stage as an API call - no model to host, a per-search price, and your passages leave your network. Shapes (check each provider's docs for current model names; not run here):

```python
import cohere                                        # Cohere Rerank
co = cohere.ClientV2()                               # reads CO_API_KEY
res = co.rerank(model="rerank-v3.5", query=q, documents=passages, top_n=5)
order = [r.index for r in res.results]               # each result has .index and .relevance_score

import voyageai                                      # Voyage AI
vo = voyageai.Client()                               # reads VOYAGE_API_KEY
res = vo.rerank(q, passages, model="rerank-2", top_k=5)
order = [r.index for r in res.results]

import httpx                                         # Jina AI (plain HTTP)
res = httpx.post("https://api.jina.ai/v1/rerank", headers={"Authorization": f"Bearer {JINA_API_KEY}"},
                 json={"model": "jina-reranker-v2-base-multilingual", "query": q, "documents": passages, "top_n": 5})
order = [r["index"] for r in res.json()["results"]]
```

Choose a hosted reranker for multilingual quality and zero ops; a local cross-encoder (like the one above, or `BAAI/bge-reranker-*`) for data residency, cost at high QPS and predictable latency.

## Try it yourself

**Exercise 1.** Candidate depth: rerank the first-stage top 5, 10 and 20 (and all 42). How do MRR and the number of pairs scored change?

In [ ]:
# Solution — try it yourself first, then run this
for depth in [5, 10, 20, 42]:
    ranks = []
    for q in IN_SCOPE:
        cands = first_stage(q["q"]) if depth < 42 else list(range(len(CHUNKS)))
        ranks.append(first_hit(rerank(q["q"], cands[:depth], top_n=depth)[0], q))
    print(f"rerank top {depth:2d}: MRR = {mrr(ranks):.3f}, pairs scored per query <= {depth}")

**Exercise 2.** Is the cross-encoder score usable as a confidence signal? Compare the best reranker score for each answerable question with the out-of-scope one.

In [ ]:
# Solution — try it yourself first, then run this
for q in QUESTIONS:
    _, s = rerank(q["q"], first_stage(q["q"]), top_n=1)
    print(f"{s[0]:6.1f}  {q['kind']:12s} {q['q'][:60]}")
print("the out-of-scope question scores lowest, but some answerable paraphrases are not far above it:"
      " calibrate any threshold on labelled data")

**Exercise 3.** Chain them: rerank the top 20, then apply MMR to the reranked top 10 (using the section-chunk embeddings) to pick 3 diverse, relevant chunks for a two-part question.

In [ ]:
# Solution — try it yourself first, then run this
two_part = "How long do approved refunds take, and how long is the gateway warranty?"
top10, _ = rerank(two_part, first_stage(two_part), top_n=10)
sub_vecs = chunk_vecs[top10]
picks = [top10[j] for j in mmr(encode([two_part])[0], sub_vecs, k=3, lam=0.5)]
print([CHUNKS[i]["id"] for i in picks])

## Say it in an interview

- **30 seconds:** "Retrieval is two stages. A bi-encoder plus BM25 retrieves 50-200 candidates in milliseconds because passage vectors are precomputed; a cross-encoder then reads the query and each candidate together with full attention and re-sorts them, and I keep the top 3-5. It is far more accurate and far too slow for the whole corpus - cost is candidates × one forward pass - and it cannot fix recall: anything the first stage missed never reaches it. So I measure recall@candidates first and MRR / hit@1 after reranking."
- **Numbers:** 50-800 ms for 50-100 candidates depending on model and hardware; keep 3-5 chunks; MMR λ ≈ 0.5-0.7; RRF k = 60 upstream.
- **Similarity vs relevance:** embeddings measure topical closeness; a reranker is trained on "does this answer that" - the gap between the two is the whole justification for the second stage.
- **Traps:** adding a reranker to fix bad recall; reranking too many candidates and blowing the latency budget; trusting LLM-returned ids without validation; tuning MMR when the real fix is deduplication.
- **Follow-ups:** "ColBERT?" (late interaction: per-token vectors, MaxSim - between bi- and cross-encoders); "calibration?" (reranker scores are closer to calibrated than cosine; use them, plus the rank-1 vs rank-5 gap, for the 'I don't know' gate).

## Next

- [RAG pipeline from scratch](06_rag_pipeline_from_scratch.ipynb) - put retrieval, the confidence gate and generation together.
- [Advanced RAG: query rewriting, HyDE, multi-query](07_advanced_rag_query_rewriting_hyde_multiquery.ipynb) - fix the recall failures reranking cannot.
- [Sentence-transformers embeddings and cross-encoders](../10_huggingface/04_sentence_transformers_embeddings_and_cross_encoders.ipynb) · [RAG evaluation](../12_evaluation/03_rag_evaluation_retrieval_and_faithfulness.ipynb) · [Latency percentiles and cost](../16_production/05_latency_percentiles_p50_p95_and_cost.ipynb)
- Theory: [genai_flow course](../../genai_flow/index.html) · [AI system design concepts](../../ai_system_design_concepts/index.html) · [interview bank](../../ai_interview_prep/index.html)